# 01 — Document Ingestion

This notebook implements the document ingestion layer for NeuroForge. It converts raw input files
(PDF, PPTX, DOCX, images, YouTube links) into a unified `Document` model for downstream processing.

---

## PDF Loading

PDF extraction using **pdfplumber** (primary) with **PyMuPDF** (fitz) fallback.

Features:
- Page-level text extraction with page numbers
- Heading detection via font size analysis
- Multi-column layout detection and splitting
- Scanned/image-only PDF detection (routes to OCR)

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from models import Document, DocumentMetadata, Section, InputFormat
from src.ingestion.pdf_loader import PDFLoader, extract_pdf

print("Imports OK.")

### PDFLoader Usage

The `PDFLoader` class provides the main interface for PDF extraction:

In [ ]:
loader = PDFLoader()

# Update this path to point to an actual PDF file for a real test
sample_pdf_path = "../data/sample.pdf"

if Path(sample_pdf_path).exists():
    doc = loader.load(sample_pdf_path)
    print(f"Title:        {doc.metadata.title}")
    print(f"Source:       {doc.metadata.source}")
    print(f"Format:       {doc.metadata.format.value}")
    print(f"Total pages:  {doc.metadata.total_pages}")
    print(f"Sections:     {len(doc.sections)}")
    print(f"Content size: {len(doc.content)} characters")
    print()
    if doc.sections:
        print("--- Detected Headings ---")
        for s in doc.sections[:10]:
            print(f"  [L{s.level}] p.{s.page_number}: {s.heading}")
    print()
    print("--- Content Preview (first 500 chars) ---")
    print(doc.content[:500])
else:
    print(f"Sample PDF not found at: {sample_pdf_path}")
    print("To test, place a PDF file and update the path above.")
    print()
    print("--- Validation: error handling ---")
    try:
        loader.load("nonexistent.pdf")
    except FileNotFoundError as e:
        print(f"  FileNotFoundError: {e}")
    try:
        loader.load("../requirements.txt")
    except ValueError as e:
        print(f"  ValueError: {e}")
    print("\n  Error handling OK.")

### Scanned PDF Detection

The loader can detect scanned/image-only PDFs that need OCR processing:

In [ ]:
if Path(sample_pdf_path).exists():
    is_scanned = loader.is_scanned(sample_pdf_path)
    print(f"Is scanned (needs OCR): {is_scanned}")
else:
    print("No sample PDF available — skipping scanned detection demo.")
    print("The PDFLoader.is_scanned() method checks if >50% of pages have <50 chars.")

---

**PDF Loading complete.** Next sections will add PPTX, DOCX, Image/OCR, and YouTube loaders.